# Лабораторная работа № 2
## Машинное обучение на больших данных с использованием фреймворка Apache Spark и библиотеки SparkML
### Часть 1. Регрессия: случайный лес (`RandomForestRegressor`)

**Задача:** по условиям ДТП — место, время, тип дороги, погода, объекты рядом — предсказать протяжённость затора, то есть длину участка дороги, на котором ДТП затруднило движение. Такая оценка в момент сообщения подсказывает, насколько далеко предупреждать водителей и где ставить объезд.

**Вход:** таблица `sobd_accidents.us_accidents_clean` из лабораторной № 1 (вариант 2: регрессия — `RandomForestRegressor`).

### Оглавление

1. [Подготовка](#ch1)
2. [Постановка задачи](#ch2)
3. [Данные Source1](#ch3)
4. [Целевой признак](#ch4)
5. [Разделение по времени](#ch5)
6. [Конвейер модели](#ch6)
7. [Подбор гиперпараметров](#ch7)
8. [Качество на тесте](#ch8)
9. [Важность признаков](#ch9)
10. [Выводы](#ch10)

<a id="ch1"></a>
## 1. Подготовка

**Путь:** подключаем библиотеки, запускаем сессию и открываем очищенную таблицу.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

user = os.getenv("USER")

# Все пути — от папки проекта SOBD-labs: ноутбук лежит в SOBD/2024/lab1 или lab2,
# а в HDFS проект занимает папку SOBD-labs домашнего каталога (data/, warehouse/, models/)
BASE = Path.cwd().parent
HDFS_ROOT = f"hdfs:///user/{user}/SOBD-labs"

# Ресурсы. Узел кафедры (256 ядер, 188 ГБ) делят много студентов — берём ~12 %;
# на локальном стенде (4 ядра, 8 ГБ) — то, что помещается в его YARN
on_cluster = os.cpu_count() >= 64
executors, cores, memory, driver_memory = (4, 8, "6g", "4g") if on_cluster else (2, 2, "1536m", "2g")
# библиотека Iceberg: на кластере скачивается из Maven, на стенде уже лежит среди jar-файлов Spark
iceberg_package = "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.0" if on_cluster else ""

spark = (
    SparkSession.builder
    .appName("lab2-part1-random-forest")
    .master("yarn")
    .config("spark.dynamicAllocation.enabled", "false")  # ровно столько исполнителей, сколько задали
    .config("spark.executor.instances", executors)
    .config("spark.executor.cores", cores)
    .config("spark.executor.memory", memory)
    .config("spark.driver.memory", driver_memory)
    .config("spark.sql.shuffle.partitions", 4 * executors * cores)  # 4 раздела на ядро вместо 200
    # Iceberg: таблицы и их метаданные — в HDFS
    .config("spark.jars.packages", iceberg_package)
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.spark_catalog", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.spark_catalog.type", "hadoop")
    .config("spark.sql.catalog.spark_catalog.warehouse", f"{HDFS_ROOT}/warehouse")
    # без кеша каталога: после перезаписи таблицы сессия сразу видит новую версию
    .config("spark.sql.catalog.spark_catalog.cache-enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark.catalog.setCurrentDatabase("sobd_accidents")

sns.set_theme(style="whitegrid")
FIGURES = BASE / "figures"  # SOBD/2024/figures
FIGURES.mkdir(exist_ok=True)


def save(name):
    """Сохраняет текущий рисунок для главы курсовой."""
    plt.savefig(f"{FIGURES}/{name}.png", dpi=150, bbox_inches="tight")

In [ ]:
clean = spark.table("us_accidents_clean")
print(f"ДТП в очищенной таблице: {clean.count():,}")

<a id="ch2"></a>
## 2. Постановка задачи

**Целевой признак** — `distance_mi`, протяжённость затора в милях.

**Данные:** только `Source1`. Разведочный анализ (лаб. № 1, глава 10.5) показал, что `Source2` протяжённость почти не измеряет — у него она почти всегда 0 или ровно 0,01 мили.

**Признаки** — только то, что известно в момент сообщения о ДТП:

| Группа | Признаки |
| --- | --- |
| Категориальные | `state`, `road_type`, `weather_group`, `sunrise_sunset` |
| Время | `hour`, `weekday`, `month` |
| Погода | `temperature_f`, `humidity_pct`, `pressure_in`, `visibility_mi`, `wind_speed_mph` |
| Место | `start_lat`, `start_lng` |
| Объекты рядом | 12 бинарных признаков |

**Не используем:** `severity` и `duration_min` — это тоже последствия ДТП, известные позже; `year` — тестируем на будущем периоде.

**Метрики:** RMSE, MAE и $R^2$.

<a id="ch3"></a>
## 3. Данные Source1

**Путь:** берём `Source1` → проверяем, одинаково ли он измерял протяжённость в разные годы.

In [ ]:
s1 = clean.filter(F.col("source") == "Source1")

(s1.groupBy("year")
   .agg(F.count("*").alias("records"),
        F.expr("percentile_approx(distance_mi, 0.5)").alias("median_distance"),
        F.avg((F.col("distance_mi") == 0).cast("int")).alias("zero_share"))
   .orderBy("year")
   .toPandas().round(3))

В 2019–2020 годах у `Source1` 40 % нулей, а медиана в пять-шесть раз ниже, чем в остальные годы: протяжённость тогда записывали иначе. С 2021 года картина стабильна, и записей хватает с запасом. Берём 2021 — март 2023.

In [ ]:
data = s1.filter(F.col("year") >= 2021)
print(f"Записей Source1 за 2021–2023: {data.count():,}")

<a id="ch4"></a>
## 4. Целевой признак

**Путь:** смотрим распределение протяжённости → выбираем, в какой шкале её предсказывать.

In [ ]:
(data.select("distance_mi")
     .summary("min", "1%", "25%", "50%", "75%", "99%", "max")
     .toPandas().set_index("summary").astype(float).T)

Протяжённость меняется от тысячных долей мили до сотен миль. Ошибка в одну милю для затора длиной 0,1 и 20 миль — совсем разные промахи. Поэтому учим модель на десятичном логарифме: тогда ошибка 0,3 — это промах в два раза в любую сторону, независимо от длины затора.

Нулевых значений около сотни на 2,8 млн, логарифма у них нет — убираем.

In [ ]:
data = (
    data.filter(F.col("distance_mi") > 0)
    .withColumn("log_distance", F.log10("distance_mi"))
)

sample = data.select("log_distance").sample(0.01, seed=42).toPandas()
sample["log_distance"].hist(bins=60, figsize=(8, 4))
plt.title("Распределение lg(протяжённость, мили), выборка 1 %")
save("rf_target_hist")
plt.show()

В логарифмической шкале распределение близко к колоколу: почти все значения между −2 и 1, медиана около −0,46 (0,34 мили). Два горба — это короткие заторы на улицах и длинные на магистралях.

<a id="ch5"></a>
## 5. Разделение по времени

**Путь:** модель будет оценивать новые ДТП, поэтому учим её на первых полутора годах, а проверяем на следующих девяти месяцах.

In [ ]:
train = data.filter(F.col("start_time") < "2022-07-01")
test = data.filter(F.col("start_time") >= "2022-07-01")

print(f"Обучение (2021 — июнь 2022): {train.count():,}")
print(f"Тест (июль 2022 — март 2023): {test.count():,}")

<a id="ch6"></a>
## 6. Конвейер модели

**Путь:** собираем конвейер из трёх шагов:

1. `StringIndexer` — категории → номера;
2. `VectorAssembler` — все признаки в один вектор;
3. `RandomForestRegressor` — случайный лес.

Деревьям не нужны ни масштабирование, ни one-hot кодирование: категорию лес делит на группы сам. Самая многозначная категория — штат (49 значений), поэтому число корзин `maxBins` берём с запасом — 64.

In [ ]:
categorical = ["state", "road_type", "weather_group", "sunrise_sunset"]
indexed = [f"{c}_index" for c in categorical]
numeric = ["hour", "weekday", "month", "temperature_f", "humidity_pct", "pressure_in",
           "visibility_mi", "wind_speed_mph", "start_lat", "start_lng"]
poi = ["amenity", "bump", "crossing", "give_way", "junction", "no_exit",
       "railway", "roundabout", "station", "stop", "traffic_calming", "traffic_signal"]
feature_cols = indexed + numeric + poi

indexer = StringIndexer(inputCols=categorical, outputCols=indexed, handleInvalid="keep")
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
forest = RandomForestRegressor(featuresCol="features", labelCol="log_distance", maxBins=64, seed=42)

pipeline = Pipeline(stages=[indexer, assembler, forest])

<a id="ch7"></a>
## 7. Подбор гиперпараметров

**Путь:** перебираем глубину деревьев и их число кросс-валидацией на трёх блоках, критерий — RMSE.

Перебор — это 18 обучений леса, поэтому подбираем на 5 % обучающей выборки (около 90 тысяч записей).

In [ ]:
grid = (
    ParamGridBuilder()
    .addGrid(forest.maxDepth, [6, 10, 14])
    .addGrid(forest.numTrees, [20, 40])
    .build()
)

rmse = RegressionEvaluator(labelCol="log_distance", predictionCol="prediction", metricName="rmse")

# parallelism=2: на небольшой выборке ядра простаивают, поэтому учим по две модели одновременно
cv = CrossValidator(estimator=pipeline, estimatorParamMaps=grid, evaluator=rmse,
                    numFolds=3, parallelism=2, seed=42)
cv_model = cv.fit(train.sample(0.05, seed=42).cache())

results = pd.DataFrame([
    {"maxDepth": params[forest.maxDepth], "numTrees": params[forest.numTrees], "rmse": metric}
    for params, metric in zip(grid, cv_model.avgMetrics)
])
results.sort_values("rmse").round(4)

Финальную модель с лучшими параметрами учим на 30 % обучающей выборки — около 540 тысяч записей. Все 1,8 млн на стенде с 8 ГБ не помещаются в память исполнителей: данные уходят на диск, и обучение растягивается больше чем на полчаса. На кластере кафедры можно взять выборку целиком.

In [ ]:
best_params = grid[results["rmse"].idxmin()]
model = pipeline.fit(train.sample(0.3, seed=42).cache(), best_params)

best_forest = model.stages[-1]
print(f"maxDepth = {best_forest.getMaxDepth()}, numTrees = {best_forest.getNumTrees}")

<a id="ch8"></a>
## 8. Качество на тесте

**Путь:** считаем метрики модели на тестовом периоде → сравниваем с двумя простыми прогнозами без модели:

* **среднее** — для всех ДТП одно и то же среднее значение из обучения;
* **правило по типу дороги** — среднее из обучения для своего типа дороги (самый сильный фактор из разведочного анализа).

Модель полезна, только если обгоняет оба.

In [ ]:
train_mean = train.agg(F.avg("log_distance")).first()[0]
road_rule = train.groupBy("road_type").agg(F.avg("log_distance").alias("road_rule"))

predictions = (
    model.transform(test)
    .withColumn("train_mean", F.lit(train_mean))
    .join(road_rule, "road_type")
    .cache()
)


def metrics(prediction_col):
    result = {}
    for name in ["rmse", "mae", "r2"]:
        evaluator = RegressionEvaluator(labelCol="log_distance", predictionCol=prediction_col, metricName=name)
        result[name] = evaluator.evaluate(predictions)
    return result


comparison = pd.DataFrame({
    "случайный лес": metrics("prediction"),
    "правило по типу дороги": metrics("road_rule"),
    "среднее": metrics("train_mean"),
}).T
comparison["ошибка в разах (10^MAE)"] = 10 ** comparison["mae"]
comparison.round(3)

Лес обгоняет оба простых прогноза: $R^2$ = 0,34 против 0,24 у правила по типу дороги. В среднем он ошибается в 2,9 раза в ту или другую сторону, правило — в 3,2, одно среднее — в 4,1.

На тесте ошибка выше, чем на кросс-валидации (RMSE 0,60 против 0,51): во второй половине 2022 года заторы в среднем короче, чем в обучающем периоде.

Сравним прогноз и факт на 1 % тестовых записей. Точки по диагонали — точные прогнозы.

In [ ]:
sample = predictions.select("log_distance", "prediction").sample(0.01, seed=42).toPandas()

plt.figure(figsize=(5, 5))
plt.scatter(sample["log_distance"], sample["prediction"], s=3, alpha=0.2)
plt.plot([-3, 2], [-3, 2], color="gray", linestyle="--")
plt.xlabel("Факт, lg(мили)")
plt.ylabel("Прогноз, lg(мили)")
plt.title("Прогноз против факта")
save("rf_scatter")
plt.show()

Прогнозы ложатся в две полосы: около 0,15 мили для улиц и около 1 мили для магистралей. Внутри одного типа дороги модель различает заторы слабо и сглаживает крайние значения к середине.

Средний прогноз по типам дорог против фактического, в милях (среднее логарифмов переводим обратно возведением 10 в степень): видно, различает ли модель короткие и длинные заторы.

In [ ]:
by_road = (
    predictions.groupBy("road_type")
    .agg(F.avg("log_distance").alias("факт"), F.avg("prediction").alias("прогноз"))
    .toPandas()
    .set_index("road_type")
)
(10 ** by_road).round(2)

Модель верно различает короткие заторы на улицах и в семь раз более длинные на межштатных магистралях, но на тесте немного завышает протяжённость по всем типам дорог — это тот же сдвиг второй половины 2022 года.

<a id="ch9"></a>
## 9. Важность признаков

In [ ]:
importance = pd.Series(best_forest.featureImportances.toArray(), index=feature_cols).sort_values()

importance.tail(15).plot.barh(figsize=(7, 5), title="15 самых важных признаков случайного леса")
save("rf_importance")
plt.show()

<a id="ch10"></a>
## 10. Выводы

Тип дороги даёт почти 60 % важности — как и в разведочном анализе. Дальше идут штат, координаты и объекты рядом (светофор, переход), которые отличают городские улицы. Погода и время суток на протяжённость затора почти не влияют.

**Итоги**

* Задача — по условиям ДТП предсказать протяжённость затора. Данные — `Source1` за 2021 — март 2023 (2,8 млн записей): только он её измеряет, а в 2019–2020 годах измерял иначе.
* Протяжённость меняется на пять порядков, поэтому модель учится на её десятичном логарифме.
* Конвейер `SparkML`: `StringIndexer` → `VectorAssembler` → `RandomForestRegressor`, 26 признаков. Кросс-валидация на 5 % обучающей выборки выбрала 40 деревьев глубиной 14; финальная модель обучена на 30 % (около 540 тыс. записей) — ограничение памяти стенда.
* На тестовом периоде (июль 2022 — март 2023, 1 млн записей) $R^2$ = 0,34, типичная ошибка — в 2,9 раза. Это лучше правила «среднее по типу дороги» ($R^2$ = 0,24) и прогноза одним средним ($R^2$ = 0).
* Ограничения: внутри типа дороги модель различает заторы слабо, а часть разброса — особенности измерения у источника. Улучшить прогноз могли бы признаки, которых в датасете нет: интенсивность движения, число полос, ограничение скорости.

In [ ]:
spark.stop()